# Advanced processing of Strands Agents Response

Strands Agents allows you to intercept and process events as they happen during agent execution using two methods: 
    
- **Async iterators**: ideal for asynchronous frameworks like FastAPI, aiohttp, or Django Channels. For these environments, the SDK offers the `stream_async` method which returns an asynchronous iterator. 
- **Callback handlers**: allow you to intercept and process events as they happen during agent execution. This enables real-time monitoring, custom output formatting, and integration with external systems.

In this example, we will show you how to use both methods to handle calls on your agent


## Agent Details
<div style="float: left; margin-right: 20px; ">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|Feature used        |async iterators, callback handlers                 |
|Agent Structure     |single agent architecture                          |
|Native tools used   |calculator                                         |
|Custom tools created|Weather forecast                                   |

</div>

## Architecture

<div style="text-align:left;">
    <img src="images/architecture.png" width="65%" />
</div>

## Key Features
* Async Iterators for Streaming
* Callback Handlers


## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock

Let's now install the requirement packages for our Strands Agent Agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [1]:
import asyncio

import httpx
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from strands import Agent, tool
from strands_tools import calculator

## Method 1 - Async Iterators for Streaming


Strands Agents provides support for asynchronous iterators through the `stream_async` method, enabling real-time streaming of agent responses in asynchronous environments like web servers, APIs, and other async applications.

Since we are show casing this example in a notebook, we need to apply `nest_asyncio` to allow nested use of `asyncio.run` and `loop.run_until_complete`

In [2]:
nest_asyncio.apply()

### Creating and invoking agent with stream_async

Let's now create our agent with a built-in calculator tool and no `callback_handler`. We will use the `stream_async` method to iteract over the streamed agent events

In [3]:
# Initialize our agent without a callback handler
agent = Agent(
    model="amazon.nova-pro-v1:0",  # Optional: Specify the model ID
    tools=[calculator], 
    callback_handler=None)

# Async function that iterators over streamed agent events


async def process_streaming_response():
    agent_stream = agent.stream_async("Calculate 2+2")
    async for event in agent_stream:
        print(event)


# Run the agent
asyncio.run(process_streaming_response())

{'init_event_loop': True}
{'start': True}
{'start_event_loop': True}
{'event': {'messageStart': {'role': 'assistant'}}}
{'event': {'contentBlockDelta': {'delta': {'text': '<thinking'}, 'contentBlockIndex': 0}}}
{'data': '<thinking', 'delta': {'text': '<thinking'}, 'agent': <strands.agent.agent.Agent object at 0x11443ea50>, 'event_loop_cycle_id': UUID('56d3ac7e-911d-457b-80f5-fea0bdb52f00'), 'request_state': {}, 'event_loop_cycle_trace': <strands.telemetry.metrics.Trace object at 0x114ca7b60>, 'event_loop_cycle_span': NonRecordingSpan(SpanContext(trace_id=0x00000000000000000000000000000000, span_id=0x0000000000000000, trace_flags=0x00, trace_state=[], is_remote=False))}
{'event': {'contentBlockDelta': {'delta': {'text': '>'}, 'contentBlockIndex': 0}}}
{'data': '>', 'delta': {'text': '>'}, 'agent': <strands.agent.agent.Agent object at 0x11443ea50>, 'event_loop_cycle_id': UUID('56d3ac7e-911d-457b-80f5-fea0bdb52f00'), 'request_state': {}, 'event_loop_cycle_trace': <strands.telemetry.metric

╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 2+2                 │                                                                            │
│  │ Result    │ 4                   │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'message': {'role': 'user', 'content': [{'toolResult': {'status': 'success', 'content': [{'text': 'Result: 4'}], 'toolUseId': 'tooluse_Fbc0caIxQSG1v3OnVgaIMB'}}]}}
{'start': True}
{'start': True}
{'start_event_loop': True}
{'event': {'messageStart': {'role': 'assistant'}}}
{'event': {'contentBlockDelta': {'delta': {'text': 'The'}, 'contentBlockIndex': 0}}}
{'data': 'The', 'delta': {'text': 'The'}, 'agent': <strands.agent.agent.Agent object at 0x11443ea50>, 'event_loop_cycle_id': UUID('746028e1-84c4-42d2-8851-68ddd85b0918'), 'request_state': {}, 'event_loop_cycle_trace': <strands.telemetry.metrics.Trace object at 0x114cd10f0>, 'event_loop_cycle_span': NonRecordingSpan(SpanContext(trace_id=0x00000000000000000000000000000000, span_id=0x0000000000000000, trace_flags=0x00, trace_state=[], is_remote=False)), 'model': <strands.models.bedrock.BedrockModel object at 0x11443eba0>, 'messages': [{'role': 'user', 'content': [{'text': 'Calculate 2+2'}]}, {'role': 'assistant', 'content': [{'text': '

###  Tracking event loop lifecycle

This example illustrates the event loop lifecycle and how events relate to each other. It's useful for understanding the flow of execution in the Strands Agent:

Let's create some printing format code to better analyse the agent stream events. We will continue to use the same agent for it

In [4]:
# Async function that iterators over streamed agent events


async def process_streaming_response():
    agent_stream = agent.stream_async("What is the capital of France and what is 42+7?")
    async for event in agent_stream:
        # Track event loop lifecycle
        if event.get("init_event_loop", False):
            print("🔄 Event loop initialized")
        elif event.get("start_event_loop", False):
            print("▶️ Event loop cycle starting")
        elif event.get("start", False):
            print("📝 New cycle started")
        elif "message" in event:
            print(f"📬 New message created: {event['message']['role']}")
        elif event.get("force_stop", False):
            print(
                f"🛑 Event loop force-stopped: {event.get('force_stop_reason', 'unknown reason')}"
            )

        # Track tool usage
        if "current_tool_use" in event and event["current_tool_use"].get("name"):
            tool_name = event["current_tool_use"]["name"]
            print(f"🔧 Using tool: {tool_name}")

        # Show only a snippet of text to keep output clean
        if "data" in event:
            # Only show first 20 chars of each chunk for demo purposes
            data_snippet = event["data"][:20] + (
                "..." if len(event["data"]) > 20 else ""
            )
            print(f"📟 Text: {data_snippet}")

    return event["result"]


# Run the agent
asyncio.run(process_streaming_response())

🔄 Event loop initialized
📝 New cycle started
▶️ Event loop cycle starting
📟 Text: <thinking
📟 Text: >
📟 Text:  The
📟 Text:  user
📟 Text:  has
📟 Text:  asked
📟 Text:  two
📟 Text:  separate
📟 Text:  questions
📟 Text: :
📟 Text:  one
📟 Text:  about
📟 Text:  a
📟 Text:  geographical
📟 Text:  fact
📟 Text:  and
📟 Text:  another
📟 Text:  about
📟 Text:  an
📟 Text:  arithmetic
📟 Text:  calculation
📟 Text: .
📟 Text:  I
📟 Text:  can
📟 Text:  use
📟 Text:  the
📟 Text:  calculator
📟 Text:  tool
📟 Text:  to
📟 Text:  perform
📟 Text:  the
📟 Text:  arithmetic
📟 Text:  calculation
📟 Text: ,
📟 Text:  but
📟 Text:  I
📟 Text:  will
📟 Text:  need
📟 Text:  to
📟 Text:  provide
📟 Text:  the
📟 Text:  geographical
📟 Text:  fact
📟 Text:  from
📟 Text:  my
📟 Text:  existing
📟 Text:  knowledge
📟 Text: .
📟 Text:  </
📟 Text: thinking
📟 Text: >


📟 Text: The
📟 Text:  capital
📟 Text:  of
📟 Text:  France
📟 Text:  is
📟 Text:  Paris
📟 Text: .
📟 Text: 
📟 Text:  


🔧 Using tool: calculator
📬 New message created: assistant


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 42+7                │                                                                            │
│  │ Result    │ 49                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📬 New message created: user
📝 New cycle started
📝 New cycle started
▶️ Event loop cycle starting
📟 Text: The
📟 Text:  capital
📟 Text:  of
📟 Text:  France
📟 Text:  is
📟 Text:  Paris
📟 Text: .
📟 Text:  The
📟 Text:  result
📟 Text:  of
📟 Text:  the
📟 Text:  calculation
📟 Text: 
📟 Text:  4
📟 Text: 2
📟 Text: +
📟 Text: 7
📟 Text:  is
📟 Text: 
📟 Text:  4
📟 Text: 9
📟 Text: .
📟 Text: 
📬 New message created: assistant


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The capital of France is Paris. The result of the calculation 42+7 is 49.'}], 'metadata': {'usage': {'inputTokens': 1650, 'outputTokens': 23, 'totalTokens': 1673}, 'metrics': {'latencyMs': 559, 'timeToFirstByteMs': 622}}}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_fYOVeSH5Cv2UbacGEpc64a', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': '42+7'}}, call_count=2, success_count=2, error_count=0, total_time=0.008374214172363281)}, cycle_durations=[4.902293920516968, 0.6702659130096436, 1.3649070262908936, 0.8353443145751953], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='56d3ac7e-911d-457b-80f5-fea0bdb52f00', usage={'inputTokens': 1408, 'outputTokens': 67, 'totalTokens': 1475}), EventLoopCycleMetric(event_loop_cycle_id='746028e1-84c4-42d2-8851-68ddd85b0918', usage={'inputTokens': 1505

### FastAPI Integration

You can also integrate your `stream_async` with FastAPI to create a streaming endpoint to your applications. For this, we will add a `weather_forecast` tool to our agent. The architecture update looks as following

<div style="text-align:left;">
    <img src="images/architecture_2.png" width="65%" />
</div>

In [5]:
# Tool definition


@tool
def weather_forecast(city: str, days: int = 3) -> str:
    return f"Weather forecast for {city} for the next {days} days..."


# FastAPI app
app = FastAPI()


class PromptRequest(BaseModel):
    prompt: str


@app.post("/stream")
async def stream_response(request: PromptRequest):
    async def generate():
        agent = Agent(tools=[calculator, weather_forecast], callback_handler=None)
        try:
            async for event in agent.stream_async(request.prompt):
                if "data" in event:
                    yield event["data"]
        except Exception as e:
            yield f"Error: {str(e)}"

    return StreamingResponse(generate(), media_type="text/plain")


# Function to start server without blocking


async def start_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8001, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()


# Run server as background task
if "server_task" not in globals():
    server_task = asyncio.create_task(start_server())
    await asyncio.sleep(0.1)  # Give server time to start

print("✅ Server is running at http://0.0.0.0:8001")

INFO:     Started server process [64962]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


✅ Server is running at http://0.0.0.0:8001


INFO:     127.0.0.1:50685 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50685 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50929 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50929 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50929 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:51056 - "GET /stream HTTP/1.1" 405 Method Not Allowed
INFO:     127.0.0.1:51242 - "GET /stream HTTP/1.1" 405 Method Not Allowed


#### Invoking the FastAPI agent
And we can now invoke the agent with a prompt 

In [8]:
async def fetch_stream():
    async with httpx.AsyncClient(timeout=httpx.Timeout(45.0)) as client:
        async with client.stream(
            "POST",
            "http://0.0.0.0:8001/stream",
            json={"prompt": "What is weather in NYC?"},
        ) as response:
            async for line in response.aiter_lines():
                if line.strip():  # Skip empty lines
                    print("Received:", line)


await fetch_stream()

AsyncLibraryNotFoundError: unknown async library, or not in async context

## Method 2 - Callback Handlers for streaming

Callback handlers are a powerful feature of the Strands Agents that allow you to intercept and process events as they happen during agent execution. This enables real-time monitoring, custom output formatting, and integration with external systems.



Callback handlers receive events in real-time as they occur during an agent's lifecycle:

- Text generation from the model
- Tool selection and execution
- Reasoning process
- Errors and completions


Let's now create a custom callback handler function that formats the event inputs to highlight tool usage and model output. To do so, we will again use the agent with a calculator tool only

<div style="text-align:left;">
    <img src="images/architecture.png" width="65%" />
</div>

In [12]:
def custom_callback_handler(**kwargs):
    # Process stream data
    if "data" in kwargs:
        print(f"MODEL OUTPUT: {kwargs['data']}")
    elif "current_tool_use" in kwargs and kwargs["current_tool_use"].get("name"):
        print(f"\nUSING TOOL: {kwargs['current_tool_use']['name']}")


# Create an agent with custom callback handler
agent = Agent(   
        model="amazon.nova-pro-v1:0",  # Optional: Specify the model ID
        tools=[calculator], 
        callback_handler=custom_callback_handler)

agent("Calculate 2+2")

MODEL OUTPUT: <thinking
MODEL OUTPUT: >
MODEL OUTPUT:  The
MODEL OUTPUT:  user
MODEL OUTPUT:  has
MODEL OUTPUT:  requested
MODEL OUTPUT:  a
MODEL OUTPUT:  simple
MODEL OUTPUT:  arithmetic
MODEL OUTPUT:  calculation
MODEL OUTPUT: ,
MODEL OUTPUT:  which
MODEL OUTPUT:  can
MODEL OUTPUT:  be
MODEL OUTPUT:  performed
MODEL OUTPUT:  using
MODEL OUTPUT:  the
MODEL OUTPUT:  '
MODEL OUTPUT: evaluate
MODEL OUTPUT: '
MODEL OUTPUT:  mode
MODEL OUTPUT:  of
MODEL OUTPUT:  the
MODEL OUTPUT:  calculator
MODEL OUTPUT:  tool
MODEL OUTPUT: .
MODEL OUTPUT:  </
MODEL OUTPUT: thinking
MODEL OUTPUT: >


USING TOOL: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 2+2                 │                                                                            │
│  │ Result    │ 4                   │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

MODEL OUTPUT: The
MODEL OUTPUT:  result
MODEL OUTPUT:  of
MODEL OUTPUT:  the
MODEL OUTPUT:  calculation
MODEL OUTPUT: 
MODEL OUTPUT:  2
MODEL OUTPUT: +
MODEL OUTPUT: 2
MODEL OUTPUT:  is
MODEL OUTPUT: 
MODEL OUTPUT:  4
MODEL OUTPUT: .
MODEL OUTPUT: 


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The result of the calculation 2+2 is 4.'}], 'metadata': {'usage': {'inputTokens': 1489, 'outputTokens': 14, 'totalTokens': 1503}, 'metrics': {'latencyMs': 486, 'timeToFirstByteMs': 656}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_YlVPUCM0ol541zEqsDg5dO', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': '2+2'}}, call_count=1, success_count=1, error_count=0, total_time=0.0033011436462402344)}, cycle_durations=[1.480780839920044, 0.7566070556640625], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='f3afac5a-d5d2-4682-bfd3-b5d366463b9d', usage={'inputTokens': 1408, 'outputTokens': 51, 'totalTokens': 1459}), EventLoopCycleMetric(event_loop_cycle_id='fb676b31-37db-4cd2-9016-25879a76e484', usage={'inputTokens': 1489, 'outputTokens': 14, 'totalTokens': 1503})], usage={'inputTokens': 2897, 

### Congratulations!

In this notebook you learned how to stream your agents outputs using async iteractors and callback handlers. 